# 03 - LightGCN 嵌入空間視覺化（t-SNE）

把 LightGCN 學到的高維 embedding 降到 2D，看模型有沒有學到合理的結構。

預期：
- 讀者：性別 / 年齡相近的應該聚在一起
- 書籍：同分類大類的應該聚在一起

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import pyarrow
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import torch

from src.dataset import load_splits
from src.models.lightgcn import LightGCN, build_norm_adj

plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font='Microsoft JhengHei')

FIG = Path('../results/figures')
FIG.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

## 1. 載入模型與資料

In [ ]:
splits = load_splits()
books = pd.read_parquet('../data/processed/books.parquet')
users = pd.read_parquet('../data/processed/users.parquet')
print(f'n_users={splits.n_users}  n_items={splits.n_items}')

# 載入訓練好的 LightGCN
model = LightGCN(splits.n_users, splits.n_items, embed_dim=64, n_layers=3).to(DEVICE)
state = torch.load('../checkpoints/lightgcn_best.pt', map_location=DEVICE, weights_only=True)
model.load_state_dict(state)

train_u = torch.as_tensor(splits.train['u'].values, dtype=torch.long)
train_i = torch.as_tensor(splits.train['i'].values, dtype=torch.long)
A_hat = build_norm_adj(train_u, train_i, splits.n_users, splits.n_items, device=DEVICE)
model.set_graph(A_hat)
model.eval()

with torch.no_grad():
    user_embs, item_embs = model.propagate()
    user_embs = user_embs.cpu().numpy()
    item_embs = item_embs.cpu().numpy()

print(f'User embeddings shape: {user_embs.shape}')
print(f'Item embeddings shape: {item_embs.shape}')

## 2. 對應每個 user 與 item 的屬性（性別 / 年齡 / 分類號）

In [ ]:
# Users
inv_user = {v: k for k, v in splits.user_remap.items()}
user_meta = pd.DataFrame({'u': range(splits.n_users), 'orig': [inv_user[i] for i in range(splits.n_users)]})
user_meta = user_meta.merge(users[['user_orig', 'gender', 'age']], left_on='orig', right_on='user_orig', how='left')

def age_bucket(a):
    if pd.isna(a): return '?'
    if a < 18: return '<18'
    if a < 25: return '18-24'
    if a < 35: return '25-34'
    if a < 50: return '35-49'
    if a < 65: return '50-64'
    return '65+'
user_meta['age_bucket'] = user_meta['age'].apply(age_bucket)
user_meta.head()

In [ ]:
# Items
inv_item = {v: k for k, v in splits.item_remap.items()}
item_meta = pd.DataFrame({'i': range(splits.n_items), 'orig': [inv_item[i] for i in range(splits.n_items)]})
item_meta = item_meta.merge(books[['book_id', 'title', 'category']], left_on='orig', right_on='book_id', how='left')

category_label = {
    '0':'0 總類','1':'1 哲學','2':'2 宗教','3':'3 科學','4':'4 應用科學',
    '5':'5 社會科學','6':'6 史地中國','7':'7 史地世界','8':'8 語文文學','9':'9 藝術'
}
def cat_top(c):
    if pd.isna(c): return '?'
    s = str(c).strip()
    if s and s[0].isdigit():
        return category_label.get(s[0], '?')
    return '?'

item_meta['cat_label'] = item_meta['category'].apply(cat_top)
item_meta['cat_label'].value_counts()

## 3. t-SNE 降維

In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

# 為了速度，先抽樣 5000 個讀者、5000 本書（t-SNE 在 10K+ 點會很慢）
rng = np.random.default_rng(42)
USER_SAMPLE = 5000
ITEM_SAMPLE = 5000

u_idx = rng.choice(splits.n_users, size=USER_SAMPLE, replace=False)
i_idx = rng.choice(splits.n_items, size=ITEM_SAMPLE, replace=False)

# 先 PCA 50 dim 加速 t-SNE
print('PCA + t-SNE (users) ...')
u_pca = PCA(n_components=min(50, user_embs.shape[1])).fit_transform(user_embs[u_idx])
u_2d = TSNE(n_components=2, perplexity=30, init='pca', random_state=42, n_iter=750).fit_transform(u_pca)
print('done')

print('PCA + t-SNE (items) ...')
i_pca = PCA(n_components=min(50, item_embs.shape[1])).fit_transform(item_embs[i_idx])
i_2d = TSNE(n_components=2, perplexity=30, init='pca', random_state=42, n_iter=750).fit_transform(i_pca)
print('done')

## 4. 視覺化讀者嵌入空間（按性別 / 年齡）

In [ ]:
u_plot = pd.DataFrame({'x': u_2d[:, 0], 'y': u_2d[:, 1]})
u_plot = u_plot.join(user_meta.set_index('u').loc[u_idx, ['gender', 'age_bucket']].reset_index(drop=True))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(data=u_plot, x='x', y='y', hue='gender',
                palette={'男':'steelblue','女':'palevioletred', None:'lightgray'},
                s=8, alpha=0.6, ax=axes[0])
axes[0].set_title('LightGCN 讀者嵌入空間 (依性別)', fontsize=14)
axes[0].set_xticks([]); axes[0].set_yticks([])

age_order = ['<18','18-24','25-34','35-49','50-64','65+','?']
sns.scatterplot(data=u_plot, x='x', y='y', hue='age_bucket', hue_order=age_order,
                palette='viridis', s=8, alpha=0.6, ax=axes[1])
axes[1].set_title('LightGCN 讀者嵌入空間 (依年齡)', fontsize=14)
axes[1].set_xticks([]); axes[1].set_yticks([])
axes[1].legend(title='年齡', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(FIG / 'fig05_user_tsne.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 視覺化書籍嵌入空間（按中圖大類）

In [ ]:
i_plot = pd.DataFrame({'x': i_2d[:, 0], 'y': i_2d[:, 1]})
i_plot = i_plot.join(item_meta.set_index('i').loc[i_idx, ['cat_label']].reset_index(drop=True))

plt.figure(figsize=(12, 9))
cat_order = sorted([c for c in i_plot['cat_label'].unique() if c != '?'])
if '?' in i_plot['cat_label'].unique():
    cat_order.append('?')
sns.scatterplot(data=i_plot, x='x', y='y', hue='cat_label', hue_order=cat_order,
                palette='tab10', s=10, alpha=0.7)
plt.title('LightGCN 書籍嵌入空間 (依中圖法大類)', fontsize=14)
plt.xticks([]); plt.yticks([])
plt.legend(title='類別', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(FIG / 'fig06_item_tsne.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 訓練曲線比較

In [ ]:
import json
RESULTS = Path('../results')

histories = {}
for name in ['bprmf', 'lightgcn']:
    p = RESULTS / f'{name}_history.json'
    if p.exists():
        with open(p, encoding='utf-8') as f:
            histories[name] = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for name, h in histories.items():
    if 'history' not in h: continue
    ep = [r['epoch'] for r in h['history']]
    ls = [r['train_loss'] for r in h['history']]
    axes[0].plot(ep, ls, label=name, linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Train Loss')
axes[0].set_title('訓練 Loss 收斂曲線'); axes[0].legend()

for name, h in histories.items():
    if 'history' not in h: continue
    ep = [r['epoch'] for r in h['history'] if 'val' in r]
    rec20 = [r['val']['recall@20'] for r in h['history'] if 'val' in r]
    axes[1].plot(ep, rec20, marker='o', label=name, linewidth=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Recall@20 (val)')
axes[1].set_title('Validation Recall@20 變化'); axes[1].legend()
plt.tight_layout()
plt.savefig(FIG / 'fig07_curves.png', dpi=150)
plt.show()

## 7. 模型表現總比較表

In [ ]:
rows = []
for name in ['popular', 'itemcf', 'bprmf', 'lightgcn', 'lightgcn_si']:
    p = RESULTS / f'{name}_history.json'
    if not p.exists(): continue
    with open(p, encoding='utf-8') as f:
        h = json.load(f)
    test = h.get('test', {})
    if not test: continue
    rows.append({'Model': name, **test})

if rows:
    summary = pd.DataFrame(rows)
    cols = ['Model'] + sorted([c for c in summary.columns if c != 'Model'])
    summary = summary[cols]
    display(summary.style.format(precision=4).background_gradient(subset=summary.columns[1:], cmap='YlGn'))
    summary.to_csv('../results/summary.csv', index=False)